# Train balance bot using PPO with domain randomization

Run each cell by pressing `shift + enter`.

In [1]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

# Import custom environment
from balance_bot_env_dr import BalanceBotEnv, DomainRandomConfig

# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import PPO training module and exporter
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx
from rl.onnx_actor_to_c import export_onnx_actor_to_c

In [2]:
# Settings
MJCF_PATH = Path("../../mechanical/bala-c-plus-simplified/bala-c-plus-simplified.xml")
SEED = 42
NUM_ENVS = 4               # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 500_000    # Number of simulation steps to perform per environment
CHASSIS_TRIM_DEG = 11.4    # Natural lean (degrees) if CoM is not directly above axle

## Configure PPO and environment

In [3]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 32,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 32,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [4]:
def make_balance_bot_env(render, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path    = MJCF_PATH,
        render_mode  = "human" if render else None,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), kw=kwargs: make_balance_bot_env(render, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [5]:
def load_agent(envs, model_path):
    """
    For debugging only! Use this to load a previously trained model to skip previous phases. Note 
    that you will still need to run the cells in each prior phase that update the experiment name 
    and environment.
    """
    from rl.ppo_trainer import Agent, TrainResult

    # Make sure model_path is a Path
    model_path = Path(model_path)
    
    # Load agent from previous run
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent = Agent(envs, ppo_config).to(device)
    agent.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    agent.eval()
    print(f"Loaded model from {model_path}")
    
    # Wrap agent in a dummy result
    return TrainResult(
        agent = agent,
        checkpoint_dir = model_path.parent,
        best_model_path = model_path,
        final_model_path = None,
        best_mean_return = 0,
    )

## Phase 1: Balance only

We'll start with the easy task of only penalizing if the robot tilts (pitches) forward.

In [6]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-1"

# Create an environment that only rewards staying upright
envs = make_envs(
    NUM_ENVS,
    chassis_trim_deg=CHASSIS_TRIM_DEG,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    vel_penalty_coef=0.0,
    cmd_pos_penalty_coef=0.0,
    heading_penalty_coef=0.0,
)

In [7]:
# Choo choo train
timestamp = time.time()
result = train(ppo_config, envs=envs)
elapsed = time.time() - timestamp

# Inspect what was saved
print(f"Phase training time: {elapsed:.1f} sec")
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-1__42__1786546703
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-1
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-1__42__1786546703/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=352.35) to runs/BalanceBot-v0__balance-bot-phase-1__42__1786546703/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-1__42__1786546703/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=847.90) to runs/BalanceBot-v0__balance-bot-phase-1__42__1786546703/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-1__42__1786546703/checkpoint_iter0150.cleanrl_model
New best model saved (mean_return=1174.85) to runs/BalanceBot-v0__balance-bot-phase-1__42__1786546703/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-1__42__1786546703/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=1722.35) t

In [8]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 1735.60
Exported charts_metrics.csv (5 metrics, 4134 steps)
Exported estimator_metrics.csv (6 metrics, 1998 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 2: Penalize drift

In [9]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-2"

# Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.vel_penalty_coef=0.02
    env.vel_penalty_cap=0.25
    env.cmd_pos_penalty_coef=0.01
    env.heading_penalty_coef=0.15

In [10]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-1__42__1785964480/best_model.cleanrl_model")

In [ ]:
# Choo choo train
timestamp = time.time()
result = train(ppo_config, envs=envs, agent=result.agent)
elapsed = time.time() - timestamp

# Inspect what was saved
print(f"Phase training time: {elapsed:.1f} sec")
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-2__42__1786547984
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-2


In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 3: Domain randomization

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-3"

# Create data randomization config to add noise, delay, and deadband
dr = DomainRandomConfig(
    init_pitch_range_deg=5.0,        # recover from +/-5 deg starts
    init_pitch_rate_range=0.8,       # falls in progress
    motor_gain_range=(0.8, 1.0),     # battery sag and asymmetry (measured ~3.3-3.96V)
    pitch_noise_std_dev=0.005,       # Sensor noise
    pitch_rate_noise_std_dev=0.005,
)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-2__42__1786043803/best_model.cleanrl_model")

In [ ]:
# Choo choo train
timestamp = time.time()
result = train(ppo_config, envs=envs, agent=result.agent)
elapsed = time.time() - timestamp

# Inspect what was saved
print(f"Phase training time: {elapsed:.1f} sec")
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [17]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 1765.35
Exported charts_metrics.csv (5 metrics, 1248 steps)
Exported estimator_metrics.csv (6 metrics, 1998 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Clean up and save model

At this point, we are done with training. We want to delete the environments and save the best actor from our final training phase. We'll export this actor network as an ONNX file that can be used on a variety of hardware platforms.

In [18]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

Closing env 0
Closing env 1
Closing env 2
Closing env 3


In [19]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

/workspace/software/python/rl/ppo_trainer.py:381: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0809 23:38:48.675000 282221 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0809 23:38:48.677000 282221 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0809 23:38:48.678000 282221 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Actor exported to ONNX: runs/BalanceBot-v0__balance-bot-phase-3__42__1786317204/actor.onnx


/opt/pyenv/versions/3.12.13/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [20]:
# Export actor to .h file
export_onnx_actor_to_c(
    onnx_path   = result.checkpoint_dir / "actor.onnx",
    output_path = result.checkpoint_dir / "actor.h"
)

Weights found:
  0.weight: shape=(32, 6), dtype=float32
  0.bias: shape=(32,), dtype=float32
  2.weight: shape=(32, 32), dtype=float32
  2.bias: shape=(32,), dtype=float32
  4.weight: shape=(2, 32), dtype=float32
  4.bias: shape=(2,), dtype=float32
C header written to runs/BalanceBot-v0__balance-bot-phase-3__42__1786317204/actor.h
  Layers:  3
  Obs:     6
  Actions: 2


PosixPath('runs/BalanceBot-v0__balance-bot-phase-3__42__1786317204/actor.h')